# Colab Bootstrap — GNN-NIDS

Chạy notebook này đầu tiên trong mỗi phiên Colab. Nó sẽ:
1. Mount Google Drive (nơi chứa dữ liệu nặng + checkpoint + MLflow log).
2. Clone (lần đầu) hoặc pull (các lần sau) code từ GitHub repo.
3. Cài các thư viện trong `requirements.txt`.
4. Thêm `src/` vào `sys.path` để import module viết trong VS Code.

**Trước khi chạy:** điền `GITHUB_USER`, `GITHUB_REPO` ở cell bên dưới.

Nếu repo private: vào Colab menu trái > icon chìa khoá (Secrets) > thêm secret tên `GITHUB_TOKEN` = Personal Access Token (scope `repo`), bật "Notebook access". Không paste token trực tiếp vào cell.

In [ ]:
GITHUB_USER = "thunww"
GITHUB_REPO = "gnn-nids-tttn"

DRIVE_ROOT = "/content/drive/MyDrive/GNN-NIDS_TTTN"      # nơi lưu data/checkpoint/mlflow trên Drive
PROJECT_DIR = "/content/GNN-NIDS_TTTN"                    # nơi clone code từ GitHub về

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs(f"{DRIVE_ROOT}/data", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/mlruns", exist_ok=True)

In [ ]:
import os

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None  # repo public: không cần token

auth = f"{token}@" if token else ""
remote_url = f"https://{auth}github.com/{GITHUB_USER}/{GITHUB_REPO}.git"

if os.path.isdir(PROJECT_DIR):
    print("Repo đã tồn tại, pull bản mới nhất...")
    !cd {PROJECT_DIR} && git pull
else:
    print("Clone repo lần đầu...")
    !git clone {remote_url} {PROJECT_DIR}

In [ ]:
!pip install -q -r {PROJECT_DIR}/requirements.txt

In [ ]:
import sys
sys.path.append(f"{PROJECT_DIR}/src")

# Từ đây import module viết trong VS Code, vd:
# from etl.load import load_netflow_dataset

print("Sẵn sàng. PROJECT_DIR =", PROJECT_DIR, "| DRIVE_ROOT =", DRIVE_ROOT)

## Giải nén dữ liệu raw từ Drive

Chạy 1 lần đầu tiên (hoặc khi thêm dataset mới). Yêu cầu: đã upload các file `.zip` dữ liệu vào `MyDrive/GNN-NIDS_TTTN/data/` trước (tên file phải khớp với key trong dict `zips` bên dưới).

In [ ]:
import zipfile, os, shutil

zips = {
    "nf-cse-cic-ids2018-v2.zip": "nf-cse-cic-ids2018-v2",
    "nf-unsw-nb15-v2.zip": "nf-unsw-nb15-v2",
}

for zip_name, folder_name in zips.items():
    zip_path = f"{DRIVE_ROOT}/data/{zip_name}"
    out_dir = f"{DRIVE_ROOT}/data/raw/{folder_name}"

    if os.path.isdir(out_dir) and os.listdir(out_dir):
        print("Bỏ qua (đã có dữ liệu):", folder_name, "->", os.listdir(out_dir))
        continue

    os.makedirs(out_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(out_dir)

    # zip có thể chứa sẵn 1 thư mục con cùng tên -> đẩy phẳng ra ngoài
    nested = os.path.join(out_dir, folder_name)
    if os.path.isdir(nested):
        for fname in os.listdir(nested):
            shutil.move(os.path.join(nested, fname), os.path.join(out_dir, fname))
        os.rmdir(nested)

    print("Đã giải nén:", folder_name, "->", os.listdir(out_dir))

## Giai nen data/processed (ket qua ETL + Graph Builder tu local)

Yeu cau: da upload file processed.zip (nen ca thu muc data/processed/ o local) vao MyDrive/GNN-NIDS_TTTN/data/. Neu dat ten file khac, sua bien ZIP_NAME ben duoi.

In [ ]:
import zipfile, os, shutil

ZIP_NAME = "processed.zip"

zip_path = f"{DRIVE_ROOT}/data/{ZIP_NAME}"
out_dir = f"{DRIVE_ROOT}/data/processed"

if os.path.isdir(out_dir) and os.listdir(out_dir):
    print("Bo qua (da co du lieu):", os.listdir(out_dir))
else:
    os.makedirs(out_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(out_dir)

    # neu zip chua san 1 thu muc con "processed" -> day phang ra ngoai
    nested = os.path.join(out_dir, "processed")
    if os.path.isdir(nested):
        for fname in os.listdir(nested):
            shutil.move(os.path.join(nested, fname), os.path.join(out_dir, fname))
        os.rmdir(nested)

    print("Da giai nen:", os.listdir(out_dir))

## Chay ETL tren du lieu that

Chay sau khi da giai nen du lieu o buoc tren. Lay code moi nhat tu GitHub truoc (phong khi ban vua push sau khi mo notebook nay), roi chay pipeline ETL, luu ket qua vao Drive.

In [ ]:
!cd {PROJECT_DIR} && git pull

In [ ]:
import sys, os
sys.path.insert(0, f"{PROJECT_DIR}/src")
from etl.run_etl import run
from pathlib import Path

processed_dir = Path(f"{DRIVE_ROOT}/data/processed")
if processed_dir.exists() and any(processed_dir.iterdir()):
    print("Bo qua (data/processed da co san, khong chay lai ETL):", os.listdir(processed_dir))
else:
    run(Path(f"{DRIVE_ROOT}/data/raw"), processed_dir)

## Copy du lieu processed ve dia cuc bo (Colab local disk)

Tap train gio duoc chia shard va doc lai TU DAU moi epoch (xem `src/models/shard_graphs.py`,
`docs/decisions.md` muc 2026-07-24) -- doc truc tiep tu Drive (FUSE mount, thong luong thap,
thuong chi ~20-50MB/s) se rat cham vi phai doc lai nhieu lan. Copy 1 lan sang dia cuc bo
`/content/` (nhanh hon nhieu) truoc khi train. Checkpoint/model van duoc ghi thang len Drive
(file nho, khong cham) de khong mat tien do neu phien Colab bi ngat giua chung.

In [ ]:
import shutil, os

LOCAL_PROCESSED_DIR = "/content/data/processed"

if os.path.isdir(LOCAL_PROCESSED_DIR) and os.listdir(LOCAL_PROCESSED_DIR):
    print("Bo qua (da co ban sao cuc bo):", os.listdir(LOCAL_PROCESSED_DIR))
else:
    shutil.copytree(f"{DRIVE_ROOT}/data/processed", LOCAL_PROCESSED_DIR)
    print("Da copy xong ve", LOCAL_PROCESSED_DIR)

## Train GCN + GAT (can GPU)

Chay sau khi da co du lieu do thi (*_graphs*.pt) va da copy ve dia cuc bo (cell tren). Vao Runtime > Change runtime type > chon GPU (T4) truoc khi chay cell nay.

Doc du lieu tu `LOCAL_PROCESSED_DIR` (nhanh), ghi checkpoint/model ve Drive (`{DRIVE_ROOT}/data/processed`, an toan neu bi ngat giua chung) -- xem `train_gnn.run(processed_dir, output_dir)`.

In [ ]:
!cd {PROJECT_DIR} && python src/models/train_gnn.py {LOCAL_PROCESSED_DIR} {DRIVE_ROOT}/data/processed

## Transfer learning: CSE-CIC-IDS2018 -> UNSW-NB15 (bo sung, chay sau khi train tu dau xong)

Chi chay SAU KHI cell tren (train tu dau) da xong voi ca 2 bo -- can co san `gcn_best.pt`/`gat_best.pt`
cua CSE-CIC-IDS2018 tren Drive de nap trong so pretrain. Xem `docs/decisions.md` (muc "Transfer learning").

In [ ]:
!cd {PROJECT_DIR} && python src/models/train_gnn_transfer.py {LOCAL_PROCESSED_DIR} {DRIVE_ROOT}/data/processed